In [ ]:

# ================================================================
# HGT-ATLAS AI — COMPLETE SINGLE-CELL RESEARCH APPLICATION
# Google Colab / Jupyter Notebook
# ================================================================
# Features:
# 1. PubMed literature search and import
# 2. CSV research-data import
# 3. Biomedical entity extraction
# 4. Interactive co-mention knowledge graph
# 5. TF-IDF evidence retrieval
# 6. Candidate-link ranking
# 7. Evidence and entity explorer
# 8. CSV exports
# 9. Dataset statistics and data-quality checks
#
# IMPORTANT:
# Co-occurrence is not proof of biological interaction or HGT.
# Candidate links are exploratory rankings, not validated discoveries.
# ================================================================

%pip -q install gradio requests pandas networkx plotly scikit-learn

import re
import os
import json
import time
import tempfile
import requests
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import gradio as gr

from collections import Counter
from xml.etree import ElementTree as ET
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# --------------------------- CONFIG -------------------------------

APP_NAME = "HGT-ATLAS AI"

TERMS = [
    "horizontal gene transfer",
    "antimicrobial resistance",
    "antibiotic resistance",
    "multidrug resistance",
    "mobile genetic elements",
    "mobile genetic element",
    "plasmid-mediated resistance",
    "plasmids", "plasmid",
    "transposons", "transposon",
    "integrons", "integron",
    "bacteriophage", "conjugation",
    "transformation", "transduction",
    "Escherichia coli",
    "Klebsiella pneumoniae",
    "Acinetobacter baumannii",
    "Pseudomonas aeruginosa",
    "Staphylococcus aureus",
    "Enterococcus faecium",
    "Salmonella enterica",
    "Campylobacter jejuni",
    "Mycobacterium tuberculosis",
    "Enterobacter cloacae",
    "Vibrio cholerae",
    "Neisseria gonorrhoeae",
    "methicillin resistance",
    "carbapenem resistance",
    "vancomycin resistance",
    "colistin resistance",
    "beta-lactam resistance",
    "extended-spectrum beta-lactamase",
    "extended spectrum beta lactamase",
    "blaCTX-M", "blaNDM", "blaKPC", "blaOXA-48",
    "mecA", "vanA", "vanB", "mcr-1",
    "tetM", "ermB", "sul1", "qnrS", "intI1",
    "integrase", "transposase",
    "resistance gene", "resistance genes",
    "efflux pump", "biofilm",
    "metagenomics", "gut microbiome",
    "microbiome", "bacterial community",
    "gene transfer", "genomic island",
    "genomic islands", "mobile DNA",
    "antibiotic susceptibility", "resistome",
    "gene expression", "genome sequencing",
    "whole genome sequencing", "horizontal transfer"
]

TERMS = sorted(set(TERMS), key=len, reverse=True)

PUBMED = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

COLUMNS = [
    "id", "pmid", "title", "abstract",
    "publication_date", "source", "url"
]


# --------------------------- UTILITIES ----------------------------

def norm(x):
    return re.sub(r"\s+", " ", str(x or "")).strip()


def sentence_split(text):
    text = norm(text)
    if not text:
        return []
    return [
        s.strip()
        for s in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", text)
        if len(s.strip()) > 20
    ]


def extract_entities(text):
    text = norm(text)
    matches = []

    for term in TERMS:
        pattern = (
            r"(?<![A-Za-z0-9_-])" + re.escape(term)
            + r"(?![A-Za-z0-9_-])"
        )
        for match in re.finditer(pattern, text, re.I):
            matches.append(
                (match.start(), match.end(), term)
            )

    # Longest match wins when dictionary phrases overlap.
    selected = []
    for start, end, term in sorted(
        matches, key=lambda x: (-(x[1] - x[0]), x[0])
    ):
        if any(not (end <= a or start >= b) for a, b, _ in selected):
            continue
        selected.append((start, end, term))

    return list(dict.fromkeys(
        term for _, _, term in sorted(selected)
    ))


def normalize_record(record, idx):
    title = norm(record.get("title", ""))
    abstract = norm(record.get("abstract", ""))
    pmid = norm(record.get("pmid", ""))
    rid = norm(record.get("id", "")) or pmid or f"LOCAL-{idx:05d}"

    return {
        "id": rid,
        "pmid": pmid,
        "title": title or "Untitled research record",
        "abstract": abstract,
        "publication_date": norm(record.get("publication_date", "")),
        "source": norm(record.get("source", "User-provided")),
        "url": norm(record.get("url", "")),
    }


def unique_records(records):
    output, seen = [], set()
    for idx, record in enumerate(records or [], start=1):
        row = normalize_record(record, idx)
        key = row["pmid"] or (
            row["title"].lower(), row["abstract"][:250].lower()
        )
        if key in seen:
            continue
        seen.add(key)
        output.append(row)
    return output


def table_records(records):
    rows = []
    for r in records or []:
        text = r["title"] + " " + r["abstract"]
        rows.append([
            r["id"], r["title"], r["publication_date"],
            r["source"], len(r["abstract"]),
            len(extract_entities(text))
        ])
    return pd.DataFrame(rows, columns=[
        "Record ID", "Title", "Date",
        "Source", "Abstract characters", "Detected entities"
    ])


def status_message(records, message):
    return (
        f"### {message}\n\n"
        f"- Loaded records: **{len(records or [])}**\n"
        f"- Updated: {time.strftime('%Y-%m-%d %H:%M:%S')}"
    )


# ----------------------- PUBMED IMPORT ----------------------------

def pubmed_import(query, email, limit, records):
    records = list(records or [])
    query = norm(query)
    email = norm(email)

    if not query:
        return records, "Enter a search query.", table_records(records)

    try:
        limit = max(1, min(int(limit), 50))

        params = {
            "db": "pubmed",
            "term": query,
            "retmax": limit,
            "retmode": "json"
        }
        if email:
            params["email"] = email

        response = requests.get(
            f"{PUBMED}/esearch.fcgi",
            params=params,
            timeout=30
        )
        response.raise_for_status()
        ids = response.json().get(
            "esearchresult", {}
        ).get("idlist", [])

        if not ids:
            return records, "No matching PubMed records found.", table_records(records)

        params = {
            "db": "pubmed",
            "id": ",".join(ids),
            "retmode": "xml"
        }
        if email:
            params["email"] = email

        response = requests.get(
            f"{PUBMED}/efetch.fcgi",
            params=params,
            timeout=60
        )
        response.raise_for_status()

        root = ET.fromstring(response.text)
        existing = {r["pmid"] for r in records if r["pmid"]}
        added = 0

        for article in root.findall(".//PubmedArticle"):
            pmid = norm(article.findtext(".//MedlineCitation/PMID"))
            if not pmid or pmid in existing:
                continue

            element = article.find(".//ArticleTitle")
            title = (
                "".join(element.itertext()).strip()
                if element is not None else ""
            )

            abstract_parts = []
            for element in article.findall(".//Abstract/AbstractText"):
                text = "".join(element.itertext()).strip()
                label = element.attrib.get("Label", "")
                if text:
                    abstract_parts.append(
                        f"{label}: {text}" if label else text
                    )

            abstract = norm(" ".join(abstract_parts))

            date_node = article.find(".//PubDate")
            date = ""
            if date_node is not None:
                date = "-".join(
                    norm(date_node.findtext(part))
                    for part in ["Year", "Month", "Day"]
                    if norm(date_node.findtext(part))
                )

            records.append({
                "id": pmid,
                "pmid": pmid,
                "title": title or "Untitled publication",
                "abstract": abstract,
                "publication_date": date,
                "source": "PubMed",
                "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
            })
            existing.add(pmid)
            added += 1

        return (
            records,
            status_message(records, f"Imported {added} new PubMed records."),
            table_records(records)
        )

    except Exception as exc:
        return (
            records,
            f"Import error: {type(exc).__name__}: {str(exc)[:400]}",
            table_records(records)
        )


# ------------------------ CSV IMPORT ------------------------------

def import_csv(file, records):
    records = list(records or [])

    if file is None:
        return records, "Upload a CSV file first.", table_records(records)

    try:
        frame = pd.read_csv(file.name)
        frame.columns = [str(c).strip().lower() for c in frame.columns]

        aliases = {
            "pmid": ["pmid", "pubmed_id", "publication_id"],
            "id": ["id", "record_id", "pmid"],
            "title": ["title", "paper_title", "publication_title"],
            "abstract": ["abstract", "text", "content", "summary"],
            "publication_date": ["date", "publication_date", "year"],
            "source": ["source", "database"],
            "url": ["url", "link", "source_url"]
        }

        def get_value(row, field):
            for alias in aliases[field]:
                if alias in row.index and pd.notna(row[alias]):
                    return str(row[alias])
            return ""

        added_records = []
        for _, row in frame.iterrows():
            rec = {
                field: get_value(row, field)
                for field in aliases
            }
            rec["source"] = rec["source"] or "CSV upload"
            if rec["title"] or rec["abstract"]:
                added_records.append(rec)

        merged = unique_records(records + added_records)

        return (
            merged,
            status_message(
                merged,
                f"Imported {len(merged) - len(records)} additional record(s) from CSV."
            ),
            table_records(merged)
        )

    except Exception as exc:
        return (
            records,
            f"CSV import failed: {type(exc).__name__}: {str(exc)[:400]}",
            table_records(records)
        )


# -------------------- MANUAL RECORD IMPORT ------------------------

def add_record(title, abstract, url, records):
    records = list(records or [])
    title, abstract, url = norm(title), norm(abstract), norm(url)

    if not title and not abstract:
        return records, "Enter a title or abstract.", table_records(records)

    record = {
        "id": f"LOCAL-{int(time.time() * 1000)}",
        "pmid": "",
        "title": title or "Untitled research record",
        "abstract": abstract,
        "publication_date": "",
        "source": "User-provided",
        "url": url
    }

    records = unique_records(records + [record])

    return (
        records,
        status_message(records, "Research record added."),
        table_records(records)
    )


# ------------------------ KNOWLEDGE GRAPH -------------------------

def build_graph(records):
    graph = nx.Graph()
    evidence = {}

    for record in records or []:
        text = record["abstract"] or record["title"]

        for sentence in sentence_split(text):
            entities = extract_entities(sentence)

            for entity in entities:
                graph.add_node(entity)

            for i in range(len(entities)):
                for j in range(i + 1, len(entities)):
                    a, b = sorted((entities[i], entities[j]))
                    if a == b:
                        continue

                    if graph.has_edge(a, b):
                        graph[a][b]["weight"] += 1
                    else:
                        graph.add_edge(a, b, weight=1)

                    evidence.setdefault((a, b), []).append({
                        "record_id": record["id"],
                        "pmid": record["pmid"],
                        "title": record["title"],
                        "source": record["source"],
                        "url": record["url"],
                        "sentence": sentence
                    })

    return graph, evidence


def make_graph_plot(records):
    graph, _ = build_graph(records)

    if graph.number_of_nodes() == 0:
        fig = go.Figure()
        fig.update_layout(
            title="No entities found. Import research abstracts first.",
            template="plotly_dark",
            height=600
        )
        return fig, "No graph nodes were detected."

    if graph.number_of_nodes() > 100:
        keep = [
            node for node, _ in sorted(
                graph.degree(), key=lambda x: x[1], reverse=True
            )[:100]
        ]
        graph = graph.subgraph(keep).copy()

    pos = nx.spring_layout(graph, seed=42)
    edge_x, edge_y = [], []

    for a, b in graph.edges():
        x0, y0 = pos[a]
        x1, y1 = pos[b]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edges = go.Scatter(
        x=edge_x, y=edge_y,
        mode="lines",
        line=dict(width=1, color="rgba(140,160,180,0.45)"),
        hoverinfo="none"
    )

    node_x, node_y, labels, hover, sizes = [], [], [], [], []

    for node in graph.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        labels.append(node)
        degree = graph.degree(node)
        sizes.append(14 + min(degree * 3, 25))
        hover.append(
            f"{node}<br>Neighbors: {degree}<br>"
            "Edges indicate sentence co-occurrence only."
        )

    nodes = go.Scatter(
        x=node_x, y=node_y,
        mode="markers+text",
        text=labels,
        textposition="top center",
        hovertext=hover,
        hoverinfo="text",
        marker=dict(size=sizes, line=dict(width=1, color="white"))
    )

    fig = go.Figure([edges, nodes])
    fig.update_layout(
        title="HGT-ATLAS AI — Evidence Co-mention Graph",
        template="plotly_dark",
        showlegend=False,
        height=680,
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        margin=dict(l=10, r=10, t=50, b=35),
        annotations=[dict(
            text="Edges = co-mentions in a sentence; not proof of biological interaction.",
            xref="paper", yref="paper", x=0, y=-0.05,
            showarrow=False
        )]
    )

    summary = (
        f"**Entities:** {graph.number_of_nodes()}  \n"
        f"**Edges:** {graph.number_of_edges()}  \n"
        f"**Connected components:** {nx.number_connected_components(graph)}"
    )

    return fig, summary


# -------------------- CANDIDATE LINK RANKING ----------------------

def candidate_links(records, limit=20):
    graph, _ = build_graph(records)

    columns = [
        "Entity A", "Entity B", "Shared neighbors",
        "Adamic-Adar score", "Interpretation"
    ]

    if graph.number_of_nodes() < 3:
        return pd.DataFrame(columns=columns)

    rows = []
    try:
        for a, b, score in nx.adamic_adar_index(graph):
            if graph.has_edge(a, b):
                continue

            shared = sorted(nx.common_neighbors(graph, a, b))
            if not shared:
                continue

            rows.append({
                "Entity A": a,
                "Entity B": b,
                "Shared neighbors": ", ".join(shared[:8]),
                "Adamic-Adar score": round(float(score), 5),
                "Interpretation": "Exploratory graph candidate; requires literature review"
            })
    except Exception:
        pass

    rows.sort(key=lambda x: x["Adamic-Adar score"], reverse=True)
    return pd.DataFrame(rows[:max(1, min(int(limit), 100))], columns=columns)


def run_candidates(records, limit):
    df = candidate_links(records, limit)

    if df.empty:
        message = (
            "Not enough graph structure for candidate ranking. Import more abstracts "
            "with overlapping entities. No candidate score establishes a biological fact."
        )
    else:
        message = (
            f"Generated **{len(df)} candidate links**. A high score indicates shared "
            "graph structure only. Candidates can be known, unsupported or extraction artifacts."
        )

    return df, message


# ------------------------- EVIDENCE SEARCH ------------------------

def evidence_search(question, records, limit=5):
    question = norm(question)

    if not question:
        return "Enter a research question."

    if not records:
        return "Import research records first."

    sentences = []

    for record in records:
        text = record["abstract"] or record["title"]
        for sentence in sentence_split(text):
            sentences.append({
                "sentence": sentence,
                "title": record["title"],
                "pmid": record["pmid"],
                "url": record["url"],
                "source": record["source"]
            })

    if not sentences:
        return "No usable sentences were found in the current records."

    try:
        corpus = [r["sentence"] for r in sentences]
        vectorizer = TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            max_features=30000
        )
        matrix = vectorizer.fit_transform(corpus + [question])
        scores = cosine_similarity(matrix[-1], matrix[:-1]).ravel()
    except ValueError:
        return "Unable to retrieve evidence from the available text."

    order = scores.argsort()[::-1]
    output = [
        "## Retrieved research evidence",
        f"**Question:** {question}",
        "",
        "This is TF-IDF retrieval, not a validated scientific conclusion."
    ]
    count = 0

    for idx in order:
        score = float(scores[idx])
        if score <= 0:
            continue

        row = sentences[idx]
        citation = (
            f"[PMID {row['pmid']}]({row['url']})"
            if row["pmid"] and row["url"]
            else (
                f"[Source link]({row['url']})"
                if row["url"]
                else row["source"]
            )
        )

        output.extend([
            "",
            f"### {count + 1}. {row['title']}",
            f"- Retrieval score: `{score:.3f}`",
            f"- Source: {citation}",
            f"- Text: “{row['sentence']}”"
        ])

        count += 1
        if count >= max(1, min(int(limit), 10)):
            break

    if count == 0:
        return (
            "## No matching passages retrieved\n"
            "This does not establish that no evidence exists. Try broader terms "
            "or import more records."
        )

    output.extend([
        "",
        "---",
        "**Review before interpretation:** Check the complete source, experimental "
        "context, organism, methods, limitations and conflicting findings. "
        "The system does not independently verify the claim."
    ])

    return "\n".join(output)


# ------------------------- ENTITY EXPLORER ------------------------

def entity_report(records):
    graph, evidence = build_graph(records)

    if not records:
        return pd.DataFrame(columns=[
            "Entity", "Record mentions", "Graph degree", "Neighbors"
        ])

    mentions = Counter()
    for record in records:
        entities = extract_entities(
            record["title"] + " " + record["abstract"]
        )
        for entity in entities:
            mentions[entity] += 1

    rows = []
    for node in graph.nodes():
        rows.append({
            "Entity": node,
            "Record mentions": mentions[node],
            "Graph degree": graph.degree(node),
            "Neighbors": ", ".join(sorted(graph.neighbors(node)))
        })

    return pd.DataFrame(rows).sort_values(
        ["Record mentions", "Graph degree"], ascending=False
    ) if rows else pd.DataFrame(columns=[
        "Entity", "Record mentions", "Graph degree", "Neighbors"
    ])


# -------------------------- DATA QUALITY --------------------------

def dataset_audit(records):
    if not records:
        return pd.DataFrame([
            ["Total records", 0],
            ["Records with abstracts", 0],
            ["Records without abstracts", 0],
            ["Duplicate publication IDs", 0],
            ["Records with source URLs", 0],
            ["Entities detected", 0],
            ["Graph edges", 0]
        ], columns=["Metric", "Value"])

    df = pd.DataFrame(records)
    graph, _ = build_graph(records)
    ids = [r["pmid"] for r in records if r["pmid"]]

    metrics = [
        ["Total records", len(records)],
        ["Records with abstracts", int(df["abstract"].fillna("").str.len().gt(0).sum())],
        ["Records without abstracts", int(df["abstract"].fillna("").str.len().eq(0).sum())],
        ["Duplicate publication IDs", len(ids) - len(set(ids))],
        ["Records with source URLs", int(df["url"].fillna("").str.len().gt(0).sum())],
        ["Entities detected", graph.number_of_nodes()],
        ["Graph edges", graph.number_of_edges()]
    ]

    return pd.DataFrame(metrics, columns=["Metric", "Value"])


# ---------------------------- EXPORTS -----------------------------

def save_csv(records, kind):
    if kind == "publications":
        frame = pd.DataFrame(records or []).reindex(columns=COLUMNS)
        filename = "hgt_atlas_publications.csv"
    elif kind == "entities":
        frame = entity_report(records)
        filename = "hgt_atlas_entities.csv"
    elif kind == "candidates":
        frame = candidate_links(records, 100)
        filename = "hgt_atlas_candidate_links.csv"
    elif kind == "audit":
        frame = dataset_audit(records)
        filename = "hgt_atlas_data_audit.csv"
    else:
        raise gr.Error("Unknown export type.")

    if frame.empty:
        raise gr.Error("No data available for export.")

    path = os.path.join(tempfile.gettempdir(), filename)
    frame.to_csv(path, index=False, encoding="utf-8")
    return path


# ------------------------- DEMO DATA ------------------------------

DEMO = [
    {
        "id": "DEMO-001",
        "pmid": "",
        "title": "Synthetic demo: antimicrobial resistance",
        "abstract": (
            "This artificial software demonstration mentions antimicrobial resistance, "
            "horizontal gene transfer, plasmids and Escherichia coli. "
            "These sentences are synthetic and are not scientific findings."
        ),
        "publication_date": "",
        "source": "Synthetic demo — not a publication",
        "url": ""
    },
    {
        "id": "DEMO-002",
        "pmid": "",
        "title": "Synthetic demo: mobile genetic elements",
        "abstract": (
            "This artificial example mentions mobile genetic elements, plasmids, "
            "Klebsiella pneumoniae and antimicrobial resistance. It exists only "
            "to demonstrate the software and must not be cited as research evidence."
        ),
        "publication_date": "",
        "source": "Synthetic demo — not a publication",
        "url": ""
    },
    {
        "id": "DEMO-003",
        "pmid": "",
        "title": "Synthetic demo: bacterial communities",
        "abstract": (
            "This artificial example mentions Pseudomonas aeruginosa, antibiotic "
            "resistance, gene transfer and the microbiome. These are placeholders "
            "created for testing the application, not real scientific claims."
        ),
        "publication_date": "",
        "source": "Synthetic demo — not a publication",
        "url": ""
    }
]


# --------------------------- INTERFACE ----------------------------

with gr.Blocks(
    theme=gr.themes.Soft(),
    title=APP_NAME
) as app:

    state = gr.State([])

    gr.Markdown("""
    # 🧬 HGT-ATLAS AI
    ### Evidence-Driven Discovery Engine for Antimicrobial Resistance

    Explore research records, extract biomedical entities, visualize co-mentions,
    retrieve relevant passages and rank graph-based candidate links.

    **Integrity notice:** A graph edge is not proof of horizontal gene transfer.
    Retrieval similarity is not scientific confidence. Candidate links are hypotheses
    for investigation, not confirmed biological discoveries.
    """)

    with gr.Row():
        demo_btn = gr.Button("Load demo records")
        clear_btn = gr.Button("Clear dataset", variant="stop")
        export_main_btn = gr.Button("Export publications CSV")

    main_status = gr.Markdown("No records loaded.")
    record_table = gr.Dataframe(
        label="Research records",
        interactive=False,
        wrap=True
    )
    main_file = gr.File(label="Publications CSV")

    def load_demo():
        records = unique_records(DEMO)
        return records, status_message(records, "Synthetic demo loaded."), table_records(records)

    def clear_all():
        return [], "Dataset cleared.", pd.DataFrame(columns=[
            "Record ID", "Title", "Date", "Source",
            "Abstract characters", "Detected entities"
        ])

    demo_btn.click(
        load_demo,
        outputs=[state, main_status, record_table]
    )
    clear_btn.click(
        clear_all,
        outputs=[state, main_status, record_table]
    )
    export_main_btn.click(
        lambda r: save_csv(r, "publications"),
        inputs=[state],
        outputs=[main_file]
    )

    with gr.Tab("1 — PubMed Search"):
        gr.Markdown("Search PubMed and import titles, abstracts and publication identifiers.")
        query = gr.Textbox(
            label="PubMed query",
            value='("horizontal gene transfer" OR plasmid) AND ("antimicrobial resistance" OR "antibiotic resistance")',
            lines=2
        )
        email = gr.Textbox(label="NCBI contact email (optional)")
        limit = gr.Slider(1, 50, value=10, step=1, label="Maximum publications")
        search_btn = gr.Button("Search and import", variant="primary")
        search_status = gr.Markdown()

        search_btn.click(
            pubmed_import,
            inputs=[query, email, limit, state],
            outputs=[state, search_status, record_table]
        ).then(
            lambda r: status_message(r, "PubMed import complete."),
            inputs=[state],
            outputs=[main_status]
        )

    with gr.Tab("2 — Upload CSV"):
        gr.Markdown(
            "Upload a CSV containing columns such as `title`, `abstract`, `pmid`, "
            "`publication_date`, `source` and `url`. At least a title or abstract is needed."
        )
        csv_file = gr.File(label="Research CSV", file_types=[".csv"])
        csv_btn = gr.Button("Import CSV", variant="primary")
        csv_status = gr.Markdown()

        csv_btn.click(
            import_csv,
            inputs=[csv_file, state],
            outputs=[state, csv_status, record_table]
        ).then(
            lambda r: status_message(r, "CSV import complete."),
            inputs=[state],
            outputs=[main_status]
        )

    with gr.Tab("3 — Add Research Text"):
        manual_title = gr.Textbox(label="Publication title")
        manual_abstract = gr.Textbox(label="Abstract / research text", lines=8)
        manual_url = gr.Textbox(label="Source URL")
        manual_btn = gr.Button("Add record", variant="primary")
        manual_status = gr.Markdown()

        manual_btn.click(
            add_record,
            inputs=[manual_title, manual_abstract, manual_url, state],
            outputs=[state, manual_status, record_table]
        ).then(
            lambda r: status_message(r, "Record added."),
            inputs=[state],
            outputs=[main_status]
        )

    with gr.Tab("4 — Knowledge Graph"):
        graph_btn = gr.Button("Build / refresh graph", variant="primary")
        graph_plot = gr.Plot()
        graph_summary = gr.Markdown()
        graph_btn.click(
            make_graph_plot,
            inputs=[state],
            outputs=[graph_plot, graph_summary]
        )

    with gr.Tab("5 — Evidence Search"):
        gr.Markdown(
            "Retrieve relevant sentences from imported records. The current version "
            "uses TF-IDF and does not generate or independently verify conclusions."
        )
        question = gr.Textbox(
            label="Research question",
            lines=2,
            placeholder="What evidence discusses plasmids and antimicrobial resistance?"
        )
        evidence_limit = gr.Slider(1, 10, value=5, step=1, label="Evidence passages")
        evidence_btn = gr.Button("Retrieve evidence", variant="primary")
        evidence_output = gr.Markdown()

        evidence_btn.click(
            evidence_search,
            inputs=[question, state, evidence_limit],
            outputs=[evidence_output]
        )

    with gr.Tab("6 — Candidate Links"):
        gr.Markdown(
            "Adamic–Adar ranks pairs with shared graph neighbors. These candidates "
            "are based on the imported graph and are not validated biological predictions."
        )
        candidate_limit = gr.Slider(1, 100, value=20, step=1, label="Maximum candidates")
        candidate_btn = gr.Button("Rank candidates", variant="primary")
        candidate_status = gr.Markdown()
        candidate_table = gr.Dataframe(interactive=False, wrap=True)
        candidate_export_btn = gr.Button("Export candidates CSV")
        candidate_file = gr.File()

        candidate_btn.click(
            run_candidates,
            inputs=[state, candidate_limit],
            outputs=[candidate_table, candidate_status]
        )
        candidate_export_btn.click(
            lambda r: save_csv(r, "candidates"),
            inputs=[state],
            outputs=[candidate_file]
        )

    with gr.Tab("7 — Entity Explorer"):
        entity_btn = gr.Button("Analyze entities", variant="primary")
        entities_table = gr.Dataframe(interactive=False, wrap=True)
        entity_export_btn = gr.Button("Export entities CSV")
        entity_file = gr.File()

        entity_btn.click(
            entity_report,
            inputs=[state],
            outputs=[entities_table]
        )
        entity_export_btn.click(
            lambda r: save_csv(r, "entities"),
            inputs=[state],
            outputs=[entity_file]
        )

    with gr.Tab("8 — Dataset Audit"):
        gr.Markdown(
            "Review basic data coverage and graph statistics before interpreting results."
        )
        audit_btn = gr.Button("Run data audit", variant="primary")
        audit_table = gr.Dataframe(interactive=False)
        audit_export_btn = gr.Button("Export audit CSV")
        audit_file = gr.File()

        audit_btn.click(
            dataset_audit,
            inputs=[state],
            outputs=[audit_table]
        )
        audit_export_btn.click(
            lambda r: save_csv(r, "audit"),
            inputs=[state],
            outputs=[audit_file]
        )

    with gr.Tab("9 — Documentation"):
        gr.Markdown("""
        ## System architecture

        **Input → Extraction → Graph → Retrieval → Investigation**

        ### Current implementation
        - PubMed E-utilities import
        - CSV import and manual text ingestion
        - Dictionary-based biomedical entity extraction
        - Sentence-level co-mention graph using NetworkX
        - TF-IDF evidence retrieval
        - Adamic–Adar candidate-link ranking
        - Dataset audit and CSV exports

        ### Limitations
        - The entity dictionary is incomplete and may miss synonyms.
        - Co-mentions do not establish biological relationships.
        - Candidate-link scores are not probabilities.
        - Retrieved sentences may not support the question's premise.
        - This version does not contain a trained biomedical relation-extraction model.
        - Data is stored in the current app session, not a persistent database.
        - No authentication, multi-user access control or production deployment hardening
          is included.

        ### Responsible research
        Review original papers, preserve source metadata, respect API and publisher
        terms, annotate a gold-standard evaluation set, and report model limitations.
        Do not use this prototype for clinical decisions.
        """)

    gr.Markdown(
        "---\n"
        "**HGT-ATLAS AI** · Research prototype · Evidence first, discovery second"
    )


# ---------------------------- LAUNCH ------------------------------

print("Launching HGT-ATLAS AI...")
print("Wait for Gradio to display the application URL below.")
app.launch(share=True, debug=False)
